# Bradford Bulls — Sponsor Pricing Model

Converts raw detection data from `Sponsor_Exposure_Colab_GPU.ipynb` into **Estimated Media Value (EMV)** and
actionable **pricing recommendations** for each sponsor.

---

## Methodology

Based on industry standards from Nielsen Sports QI, API4AI CPeS framework, and ExposureEngine (arXiv 2510.04739).

### Step 1 — Quality Index (per detection)

Each detected logo in a frame receives a **Quality Index** (QI, 0–1) that captures three dimensions:

| Factor | Formula | Rationale |
|--------|---------|----------|
| **Area** | `area_frac = area_pct / 100` | Larger logo = stronger brand impression |
| **Position** | `pos = exp(-0.5 × d² / σ²)` where d = distance from frame center | Central logos command more viewer attention |
| **Clarity** | `clarity = min(1, area_frac / SAT_FRAC)` | Small logos are harder to read; saturates once legible |

```
QI = area_frac × pos_score × clarity
```

This multiplicative form (API4AI standard) ensures that a tiny, off-center, blurry logo gets near-zero QI,
while a large, central, sharp logo gets QI close to its area fraction.

### Step 2 — Effective Area-Seconds (per sponsor)

```
eff_area_sec[sponsor] = Σ  QI × Δt
```

where Δt = time each sampled frame represents (= 1 / SAMPLE_FPS seconds).

This is the **quality-adjusted equivalent of full-screen advertising seconds**.

### Step 3 — CPeS (Cost per Exposure-Second)

```
TV_30s_cost = CPM_GBP × VIEWERS_K          # £ for a 30-sec full-screen ad
CPeS        = (TV_30s_cost / 30) × DISCOUNT # £ per quality-exposure-second
```

Discount reflects that sponsorship exposure ≠ a dedicated 30-second commercial.
Nielsen benchmarks: **25–35%** of equivalent ad rate for jersey/pitch logos.

### Step 4 — Estimated Media Value (EMV)

```
EMV[sponsor] = eff_area_sec[sponsor] × CPeS
```

### Step 5 — Pricing Recommendation

```
fair_price_per_match = EMV × VALUE_RETENTION
annual_price         = fair_price_per_match × MATCHES_PER_SEASON
```

---
**References:**
- [ExposureEngine paper (arXiv 2510.04739)](https://arxiv.org/abs/2510.04739)
- [API4AI CPeS framework](https://medium.com/@API4AI/cost-per-exposure-second-pricing-the-pixel-13a14f9cada7)
- [Drive Sports Marketing — Sponsorship Exposure](https://www.drivesportsmarketing.com/calculating-sports-sponsorship-exposure/)
- [Nielsen Sports Media Valuation](https://nielsensports.com/media-valuation/)
- [Shikenso — Media Value Calculation](https://shikenso.com/blog/understanding-media-value-the-key-to-sponsorship-success)

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

In [ ]:
from pathlib import Path

# ── Input files (output from Sponsor_Exposure_Colab_GPU.ipynb) ────────────────
DETECTIONS_CSV   = '/content/drive/MyDrive/bradford_bulls/exposure_output/M05_white_1080p/detections.csv'
REPORT_CSV       = '/content/drive/MyDrive/bradford_bulls/exposure_output/M05_white_1080p/exposure_report.csv'

# ── Sampling parameter (must match what was used in the exposure notebook) ────
SAMPLE_FPS       = 1.0       # frames per second that were analysed

# ── Audience & Market parameters ──────────────────────────────────────────────
# Super League 2025: avg ~250k viewers/match on Sky Sports
# Source: footyindustry.com/superleague-record-2025
VIEWERS_K        = 250       # thousands of viewers per broadcast

# UK sports TV CPM: £15–25 per 1,000 viewers (industry benchmark)
CPM_GBP          = 18.0      # £ per 1,000 viewers for a standard ad slot

# Sponsorship discount vs. 30-sec commercial (Nielsen benchmark: 25–35%)
SPONSORSHIP_DISC = 0.30      # 30% — sponsor logo ≠ a dedicated 30-sec commercial

# ── Quality scoring parameters ────────────────────────────────────────────────
# Gaussian σ for position score (fraction of half-frame diagonal).
# σ=0.4 → logos at 40% from center get ~0.6 weight; corners get ~0.1
POS_SIGMA        = 0.4

# Area fraction at which clarity saturates (logo is 'clearly legible').
# 0.3% of frame → a typical jersey logo at medium distance
CLARITY_SAT_FRAC = 0.003     # 0.3% of frame area

# ── Pricing parameters ────────────────────────────────────────────────────────
# How much of the measured EMV to charge (reflects detection uncertainty,
# partial coverage from closeup clips only, etc.)
VALUE_RETENTION  = 0.75      # price at 75% of detected EMV

MATCHES_PER_SEASON = 27      # Super League regular season + finals

# ── Output ────────────────────────────────────────────────────────────────────
REPORT_DIR = Path('/content/drive/MyDrive/bradford_bulls/pricing_reports')
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Derived
SAMPLE_DT     = 1.0 / SAMPLE_FPS   # seconds each sampled frame represents
TV_30s_cost   = CPM_GBP * VIEWERS_K          # £ cost of a 30-sec full-screen ad
CPES          = (TV_30s_cost / 30) * SPONSORSHIP_DISC  # £ per quality-exposure-second

print(f'Detections CSV  : {DETECTIONS_CSV}')
print(f'Sample Δt       : {SAMPLE_DT:.2f}s  ({SAMPLE_FPS}fps)')
print()
print(f'=== Market Parameters ===')
print(f'Viewers         : {VIEWERS_K:,}k')
print(f'CPM             : £{CPM_GBP:.2f}')
print(f'TV 30-sec spot  : £{TV_30s_cost:,.0f}')
print(f'Discount        : {SPONSORSHIP_DISC:.0%}')
print(f'CPeS            : £{CPES:.4f} per quality-exposure-second')
print(f'  (= cost of 1 second of 100%-screen, center-frame, crystal-clear sponsor logo)')

## 3. Load detections

In [ ]:
import pandas as pd
import numpy as np

dets = pd.read_csv(DETECTIONS_CSV)

if 'sponsor' not in dets.columns:
    raise ValueError('No detections found in CSV. Run Sponsor_Exposure_Colab_GPU.ipynb first.')

# drop low-quality rows
dets = dets[dets['sponsor'].notna() & (dets['sponsor'] != '')].copy()

print(f'Loaded {len(dets):,} detections across {dets["time_s"].nunique()} sampled frames')
print(f'Sponsors found  : {sorted(dets["sponsor"].unique())}')
print()
print('Column summary:')
print(dets.describe(include='all').T[['count','mean','min','max']].to_string())

## 4. Frame-level Quality Scoring

In [ ]:
import numpy as np

def position_score(cx: float, cy: float, sigma: float = POS_SIGMA) -> float:
    """Gaussian falloff from frame center (0.5, 0.5). Center → 1.0, corners → ~0.1."""
    dx = cx - 0.5
    dy = cy - 0.5
    dist_sq = dx**2 + dy**2
    return float(np.exp(-0.5 * dist_sq / sigma**2))

def clarity_score(area_frac: float, sat_frac: float = CLARITY_SAT_FRAC) -> float:
    """Linear ramp from 0 → 1 as area_frac grows to sat_frac. Saturates at 1.0.
    Rationale: logos too small to read (<sat_frac of frame) are discounted."""
    return float(min(1.0, area_frac / sat_frac))


# ── Apply scoring ─────────────────────────────────────────────────────────────
dets['area_frac']  = dets['area_pct'] / 100.0
dets['pos_score']  = dets.apply(lambda r: position_score(r['cx'], r['cy']), axis=1)
dets['clarity']    = dets['area_frac'].apply(clarity_score)

# QI = area_frac × pos_score × clarity   (multiplicative — any weak factor kills the score)
dets['QI']         = dets['area_frac'] * dets['pos_score'] * dets['clarity']

# Contribution per detection (each sampled frame = SAMPLE_DT seconds)
dets['eff_area_s'] = dets['QI'] * SAMPLE_DT        # quality-adjusted area-seconds
dets['raw_area_s'] = dets['area_frac'] * SAMPLE_DT  # unweighted area-seconds

print('Quality scoring applied.')
print()
print(f'{"Metric":<18}  {"min":>8}  {"mean":>8}  {"max":>8}')
print('-' * 48)
for col in ['area_frac', 'pos_score', 'clarity', 'QI']:
    print(f'{col:<18}  {dets[col].min():8.4f}  {dets[col].mean():8.4f}  {dets[col].max():8.4f}')

## 5. Per-sponsor Aggregation

Aggregate all detections per sponsor into exposure metrics.

In [ ]:
agg = dets.groupby('sponsor').agg(
    n_detections   = ('QI',         'count'),
    screen_time_s  = ('QI',         lambda x: len(x) * SAMPLE_DT),   # raw seconds on screen
    raw_area_sec   = ('raw_area_s', 'sum'),   # unweighted area-seconds
    eff_area_sec   = ('eff_area_s', 'sum'),   # quality-adjusted area-seconds
    mean_QI        = ('QI',         'mean'),
    mean_area_pct  = ('area_pct',   'mean'),
    peak_area_pct  = ('area_pct',   'max'),
    mean_pos       = ('pos_score',  'mean'),
    mean_clarity   = ('clarity',    'mean'),
    mean_ocr_conf  = ('ocr_conf',   'mean'),
    first_seen_s   = ('time_s',     'min'),
    last_seen_s    = ('time_s',     'max'),
).reset_index()

# total video duration (seconds) — infer from data
video_duration_s = dets['time_s'].max() + SAMPLE_DT
agg['screen_time_pct'] = (agg['screen_time_s'] / video_duration_s * 100).round(1)

# round for readability
for col in ['screen_time_s','raw_area_sec','eff_area_sec','mean_QI',
            'mean_area_pct','peak_area_pct','mean_pos','mean_clarity','mean_ocr_conf']:
    agg[col] = agg[col].round(4)

agg = agg.sort_values('eff_area_sec', ascending=False).reset_index(drop=True)

print(f'Video duration  : {video_duration_s/60:.1f} min')
print(f'Sponsors found  : {len(agg)}')
print()
from IPython.display import display
display(agg[['sponsor','screen_time_s','screen_time_pct',
             'mean_area_pct','peak_area_pct','mean_pos','mean_QI',
             'eff_area_sec','n_detections']])

## 6. EMV Calculation

Apply CPeS to convert quality-adjusted exposure into £ Estimated Media Value.

In [ ]:
# ── EMV per sponsor ───────────────────────────────────────────────────────────
agg['EMV_GBP']       = (agg['eff_area_sec'] * CPES).round(2)

# For reference: unweighted EMV (area × time, no position/clarity discount)
agg['EMV_raw_GBP']   = (agg['raw_area_sec'] * CPES).round(2)

# Quality discount = how much QI reduces vs. raw area
agg['quality_disc_pct'] = ((1 - agg['EMV_GBP'] / agg['EMV_raw_GBP'].replace(0, np.nan)) * 100).round(1)

total_emv = agg['EMV_GBP'].sum()

print(f'=== EMV Results ===')
print(f'CPeS            : £{CPES:.4f} / quality-exposure-second')
print(f'Total EMV       : £{total_emv:,.2f}  (this match)')
print()

display(agg[['sponsor','screen_time_s','eff_area_sec',
             'EMV_raw_GBP','EMV_GBP','quality_disc_pct']].rename(columns={
    'screen_time_s':   'on-screen (s)',
    'eff_area_sec':    'eff-area-sec',
    'EMV_raw_GBP':     'EMV unweighted £',
    'EMV_GBP':         'EMV quality-adj £',
    'quality_disc_pct':'quality discount %',
}))

print()
print('Interpretation:')
print('  EMV unweighted  = raw area × time × CPeS  (no position/clarity penalty)')
print('  EMV quality-adj = applies position + clarity discount  ← recommended')
print('  quality discount % = how much the logo lost vs. ideal placement')

## 7. Pricing Recommendation

In [ ]:
pricing = agg[['sponsor','screen_time_s','EMV_GBP']].copy()

# Per-match pricing: retain a fraction of detected EMV
# (accounts for: missed detections in wide shots, partial coverage of match, etc.)
pricing['fair_price_match_GBP']  = (pricing['EMV_GBP']  * VALUE_RETENTION).round(2)

# Annualise
pricing['fair_price_season_GBP'] = (pricing['fair_price_match_GBP'] * MATCHES_PER_SEASON).round(0)

# EMV / Price ratio — shows sponsor ROI
pricing['sponsor_ROI_x']         = (pricing['EMV_GBP'] / pricing['fair_price_match_GBP'].replace(0, np.nan)).round(2)

pricing = pricing.sort_values('fair_price_season_GBP', ascending=False).reset_index(drop=True)

print('=== Pricing Recommendation ===')
print(f'Value retention : {VALUE_RETENTION:.0%}  (of detected EMV)')
print(f'Season matches  : {MATCHES_PER_SEASON}')
print()
from IPython.display import display
display(pricing.rename(columns={
    'screen_time_s':          'on-screen (s)',
    'EMV_GBP':                'EMV/match £',
    'fair_price_match_GBP':   'Price/match £',
    'fair_price_season_GBP':  'Price/season £',
    'sponsor_ROI_x':          'Sponsor ROI (x)',
}))

print()
total_season = pricing['fair_price_season_GBP'].sum()
print(f'Total sponsor revenue (fair value, season): £{total_season:,.0f}')
print()
print('Note: These are data-driven price floors based on measured exposure.')
print('Actual deal prices also factor in brand prestige, exclusivity, and negotiation.')

## 8. Sensitivity Analysis — what-if viewer count

In [ ]:
# Show how EMV changes across different audience sizes
# Useful for: live broadcast vs. highlights vs. social clips

scenarios = [
    ('Live TV (Sky Sports)',    250,  0.30),
    ('Highlights (YouTube)',    500,  0.10),  # more viewers, lower discount
    ('Social clip (<1 min)',   1500,  0.08),  # viral potential
    ('Match recording (OTT)',    80,  0.30),
]

print(f'=== Sensitivity: total EMV per scenario ===')
print(f'{"Scenario":<30}  {"Viewers (k)":>12}  {"Discount":>10}  {"Total EMV £":>14}  {"CPeS £":>10}')
print('-' * 82)

total_eff_area = agg['eff_area_sec'].sum()
for name, viewers_k, discount in scenarios:
    cpes_s = (CPM_GBP * viewers_k / 30) * discount
    emv_s  = total_eff_area * cpes_s
    print(f'{name:<30}  {viewers_k:>12,}  {discount:>10.0%}  {emv_s:>14,.2f}  {cpes_s:>10.4f}')

print()
print('Key insight: social/highlights clips often reach more viewers than live TV')
print('at lower CPM — still significant incremental exposure value.')

## 9. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Bradford Bulls — Sponsor Exposure & Pricing', fontsize=14, fontweight='bold')

palette = plt.cm.Set2(np.linspace(0, 1, len(agg)))
sponsors = agg['sponsor'].tolist()

# ── Plot 1: EMV per sponsor ───────────────────────────────────────────────────
ax = axes[0, 0]
bars = ax.barh(sponsors, agg['EMV_GBP'], color=palette)
ax.bar_label(bars, fmt='£%.2f', padding=4, fontsize=8)
ax.set_xlabel('EMV (£) per match')
ax.set_title('Estimated Media Value — quality-adjusted')
ax.invert_yaxis()

# ── Plot 2: Quality breakdown (area vs QI) ───────────────────────────────────
ax = axes[0, 1]
x = np.arange(len(sponsors))
w = 0.35
ax.bar(x - w/2, agg['raw_area_sec'],  width=w, label='Raw area-seconds', alpha=0.7, color='#4c72b0')
ax.bar(x + w/2, agg['eff_area_sec'],  width=w, label='Quality-adjusted', alpha=0.9, color='#c8102e')
ax.set_xticks(x); ax.set_xticklabels(sponsors, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Area-seconds')
ax.set_title('Raw vs Quality-adjusted Exposure')
ax.legend(fontsize=8)

# ── Plot 3: Screen time breakdown ────────────────────────────────────────────
ax = axes[1, 0]
wedges, texts, autotexts = ax.pie(
    agg['screen_time_s'],
    labels=sponsors,
    autopct='%1.1f%%',
    colors=palette,
    startangle=140,
    pctdistance=0.8,
)
for t in texts + autotexts:
    t.set_fontsize(8)
ax.set_title('Share of Screen Time')

# ── Plot 4: Quality components heatmap ───────────────────────────────────────
ax = axes[1, 1]
metrics = ['mean_area_pct', 'mean_pos', 'mean_clarity', 'mean_QI']
labels  = ['Area %', 'Position', 'Clarity', 'QI']

# normalize each metric 0-1 for heatmap
heatmap_data = agg[metrics].values.copy().astype(float)
for col_i in range(heatmap_data.shape[1]):
    col = heatmap_data[:, col_i]
    rng = col.max() - col.min()
    heatmap_data[:, col_i] = (col - col.min()) / rng if rng > 0 else col * 0

im = ax.imshow(heatmap_data, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=9)
ax.set_yticks(range(len(sponsors))); ax.set_yticklabels(sponsors, fontsize=8)
ax.set_title('Quality Factors (row-normalised)')

for i in range(len(sponsors)):
    for j in range(len(metrics)):
        raw_val = agg[metrics[j]].iloc[i]
        ax.text(j, i, f'{raw_val:.2f}', ha='center', va='center', fontsize=7,
                color='black')

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
out_png = REPORT_DIR / 'pricing_dashboard.png'
plt.savefig(out_png, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_png}')

In [ ]:
# ── Timeline: when each sponsor appears ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, max(4, len(sponsors) * 0.8)))

sponsor_list = sorted(dets['sponsor'].unique())
colors_tl    = {sp: plt.cm.Set2(i / len(sponsor_list)) for i, sp in enumerate(sponsor_list)}

for row_i, sp in enumerate(sponsor_list):
    sp_times = sorted(dets[dets['sponsor'] == sp]['time_s'].unique())
    for t in sp_times:
        ax.barh(row_i, SAMPLE_DT, left=t, height=0.6,
                color=colors_tl[sp], alpha=0.85)

ax.set_yticks(range(len(sponsor_list)))
ax.set_yticklabels(sponsor_list, fontsize=9)
ax.set_xlabel('Time (seconds)')
ax.set_title('Sponsor Appearance Timeline', fontsize=12, fontweight='bold')

def fmt_mmss(x, _):
    m, s = divmod(int(x), 60)
    return f'{m:02d}:{s:02d}'
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_mmss))
ax.xaxis.set_major_locator(mticker.MultipleLocator(60))

plt.tight_layout()
out_tl = REPORT_DIR / 'sponsor_timeline.png'
plt.savefig(out_tl, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_tl}')

## 10. Export Reports

In [ ]:
import csv
from datetime import date

today = date.today().isoformat()

# ── Full metrics CSV ──────────────────────────────────────────────────────────
full = agg.merge(pricing[['sponsor','fair_price_match_GBP',
                           'fair_price_season_GBP','sponsor_ROI_x']], on='sponsor')
full_path = REPORT_DIR / f'sponsor_pricing_{today}.csv'
full.to_csv(full_path, index=False)
print(f'Full report   → {full_path}')

# ── Executive summary CSV ─────────────────────────────────────────────────────
exec_cols = ['sponsor', 'screen_time_s', 'screen_time_pct',
             'mean_area_pct', 'mean_QI', 'eff_area_sec',
             'EMV_GBP', 'fair_price_match_GBP', 'fair_price_season_GBP']
exec_path = REPORT_DIR / f'exec_summary_{today}.csv'
full[exec_cols].to_csv(exec_path, index=False)
print(f'Exec summary  → {exec_path}')

# ── Parameters log ───────────────────────────────────────────────────────────
params_path = REPORT_DIR / f'model_params_{today}.txt'
with open(params_path, 'w') as f:
    f.write(f'Bradford Bulls Sponsor Pricing Model — {today}\n')
    f.write('=' * 50 + '\n')
    f.write(f'SAMPLE_FPS         : {SAMPLE_FPS}\n')
    f.write(f'VIEWERS_K          : {VIEWERS_K}k\n')
    f.write(f'CPM_GBP            : £{CPM_GBP}\n')
    f.write(f'SPONSORSHIP_DISC   : {SPONSORSHIP_DISC:.0%}\n')
    f.write(f'CPeS               : £{CPES:.4f}/eff-area-sec\n')
    f.write(f'POS_SIGMA          : {POS_SIGMA}\n')
    f.write(f'CLARITY_SAT_FRAC   : {CLARITY_SAT_FRAC}\n')
    f.write(f'VALUE_RETENTION    : {VALUE_RETENTION:.0%}\n')
    f.write(f'MATCHES_PER_SEASON : {MATCHES_PER_SEASON}\n')
    f.write(f'\nTotal EMV (match)  : £{agg["EMV_GBP"].sum():,.2f}\n')
    f.write(f'Total price/season : £{pricing["fair_price_season_GBP"].sum():,.0f}\n')
print(f'Model params  → {params_path}')

print()
print('All reports saved to:', REPORT_DIR)